**VGGNet** (often called **VGG**, named after the Visual Geometry Group at Oxford) is a classic Convolutional Neural Network family introduced by Karen Simonyan and Andrew Zisserman in 2014. Its defining insight is that stacking many small **3×3 convolution** layers with **2×2 max pooling** outperforms larger, irregular filters while keeping the network design uniform and modular.

---

### Core Architecture & Variants

VGGNet takes 224×224 RGB images and organizes convolutional blocks so that spatial height and width halve after each max-pooling stage while channel count doubles (typically starting at 64 up to 512). The most famous variants are **VGG-16** (16 weight layers) and **VGG-19** (19 weight layers), both ending in three fully connected layers (4,096 → 4,096 → 1,000 classes for ImageNet).

| Variant | Weight Layers | Conv Blocks Pattern | Approx. Parameters |
| --- | --- | --- | --- |
| **VGG-16** | 16 (13 Conv + 3 FC) | Five max-pool stages with 2, 2, 3, 3, and 3 conv layers | ~138M |
| **VGG-19** | 19 (16 Conv + 3 FC) | Five max-pool stages with 2, 2, 4, 4, and 4 conv layers | ~144M |

---

### Key Design Principles & Innovations

* **Uniform 3×3 Convolutions:** Two stacked 3×3 convolutions have an effective receptive field of 5×5, and three stacked 3×3 layers match a 7×7 receptive field—while adding extra ReLU non-linearities and reducing parameters compared to a single large filter.
* **Clean Modular Blocks:** Repeated `Conv3x3 -> ReLU` stacks followed by `MaxPool(2x2, stride=2)` make the architecture easy to read, scale, and adapt as a feature backbone.
* **Strong Transferability:** Features learned by VGG-16 and VGG-19 generalize well to object detection, segmentation, and neural style transfer, even though the dense classifier head makes the models heavy in memory and parameters.

---

### Role in the LeNet → AlexNet → ResNet Progression

LeNet-5 proved spatial weight sharing on small digits, and AlexNet scaled CNNs on GPUs using large kernels like 11×11 and 5×5. VGGNet standardized small 3×3 filters at greater depth, showing that depth improves representation—yet its heavy fully connected layers and lack of skip connections made even deeper plain stacks harder to train, paving the way for **ResNet** in 2015.

### 1. VGG-16 Architecture in PyTorch

VGG-16 has **13 convolutional layers** across five max-pool stages (`2, 2, 3, 3, 3` convolutions) and **3 fully connected layers** (`4096 -> 4096 -> num_classes`). Using `nn.AdaptiveAvgPool2d((7, 7))` before the classifier keeps the flattened feature size at `512 * 7 * 7 = 25,088` even if input dimensions vary slightly.

In [1]:
import torch
import torch.nn as nn


class VGG16(nn.Module):

  def __init__(self, num_classes=10, dropout=0.5):
    super().__init__()
    # Expects input shape: [Batch, 3, 224, 224]
    self.features = nn.Sequential(
        # Block 1: 224x224 -> 112x112
        nn.Conv2d(3, 64, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 2: 112x112 -> 56x56
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 3: 56x56 -> 28x28
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 4: 28x28 -> 14x14
        nn.Conv2d(256, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 5: 14x14 -> 7x7
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

    self.avgpool = nn.AdaptiveAvgPool2d((7, 7))

    self.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout),
        nn.Linear(4096, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout),
        nn.Linear(4096, num_classes),
    )

  def forward(self, x):
    x = self.features(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.classifier(x)
    return x

### 2. Training Script on CIFAR-10 with Logs & Graphs

This script resizes CIFAR-10 images to 224×224, trains `VGG16`, prints batch and epoch logs, and saves training loss and test accuracy curves.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Training VGG-16 on device: {device}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model = VGG16(num_classes=10, dropout=0.5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4
)

num_epochs = 10
train_losses, test_accuracies = [], []

print("\n--- Starting VGG-16 Training ---")
for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0

  for batch_idx, (images, targets) in enumerate(train_loader):
    images, targets = images.to(device), targets.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    if (batch_idx + 1) % 200 == 0:
      print(
          f"Epoch [{epoch+1}/{num_epochs}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss.item():.4f}"
      )

  epoch_loss = running_loss / len(train_dataset)
  train_losses.append(epoch_loss)

  model.eval()
  correct, total = 0, 0
  with torch.no_grad():
    for images, targets in test_loader:
      images, targets = images.to(device), targets.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs, 1)
      total += targets.size(0)
      correct += (predicted == targets).sum().item()

  test_acc = (correct / total) * 100
  test_accuracies.append(test_acc)
  print(
      f"--> Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_loss:.4f} | Test Acc: {test_acc:.2f}%"
  )

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(
    range(1, num_epochs + 1),
    train_losses,
    marker="o",
    color="navy",
    label="Train Loss",
)
plt.title("VGG-16 Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(
    range(1, num_epochs + 1),
    test_accuracies,
    marker="s",
    color="darkorange",
    label="Test Accuracy",
)
plt.title("VGG-16 Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig("vgg16_cifar10_metrics.png")
print("[INFO] Saved plot to 'vgg16_cifar10_metrics.png'")
plt.show()

[INFO] Training VGG-16 on device: cuda


100%|██████████| 170M/170M [25:12<00:00, 113kB/s]



--- Starting VGG-16 Training ---
Epoch [1/10] | Batch [200/782] | Loss: 2.2975
Epoch [1/10] | Batch [400/782] | Loss: 2.3045
Epoch [1/10] | Batch [600/782] | Loss: 2.3043
--> Epoch [1/10] | Train Loss: 2.3031 | Test Acc: 10.00%
Epoch [2/10] | Batch [200/782] | Loss: 2.3042
Epoch [2/10] | Batch [400/782] | Loss: 2.2960
Epoch [2/10] | Batch [600/782] | Loss: 2.3052


---

### 3. Practical Notes for VGG-16

* **High Memory & Compute Cost:** VGG-16 uses ~138 million parameters—most sitting in the `25,088 -> 4096` and `4096 -> 4096` linear layers. Use a smaller batch size (such as `32` or `64`) or mixed precision (`torch.amp.autocast`) if you hit GPU memory limits.
* **BatchNorm Variant:** Plain VGG-16 can be sensitive to learning rate and weight initialization when trained from scratch. In practice, `models.vgg16_bn(weights=None)` or pretrained `models.vgg16(weights=models.VGG16_Weights.DEFAULT)` converges much more reliably.